In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import random
from tokenizer import Tokenizer

In [2]:
with open('shake_spear.txt','r+',encoding='utf-8') as file:
    data = file.read()

In [3]:
tokens = [j for  i in data for j in i.lower().split()]
counter = Counter(tokens)#frequncy map of words
vocab = {word: i+2 for i, word in enumerate(counter)}#maping words to int (token) for encoding
vocab["<pad>"] = 0 
vocab["<unk>"] = 1
inv_vocab = {i: w for w, i in vocab.items()}#maping int to words for decoding


In [6]:
def encode(input:str):
    return [vocab[j] for i in input for j in i.lower().split()]

encoded_data =  encode(input=data) 
train_X, train_Y = [], []
for i in range(len(encoded_data)-5):
    train_X.append(encoded_data[:i+5])
    train_Y.append(encoded_data[i])
    
train_Y = torch.tensor(train_Y)
train_X = torch.tensor(train_X)
print("Training samples:", train_X.shape)


KeyboardInterrupt: 

In [53]:
class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, nhead=2, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, src):
        # src: [batch_size, seq_len]
        src = self.embedding(src)  # [batch, seq_len, embed_dim]
        src = src.permute(1, 0, 2)  # Transformer expects [seq_len, batch, embed_dim]
        output = self.transformer(src)
        output = self.fc_out(output[-1])  # Use last token output
        return output

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniTransformer(vocab_size=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in zip(train_X, train_Y):
        x, y = x.unsqueeze(0).to(device), y.unsqueeze(0).to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_X):.4f}")
